# Confidence score algorithm — reproducing the Validation Framework worked example

**Source deck**: `docs/Validation framework.pdf` — working session,
UNESWA / MET / TWG / NDRMA, **3 July 2026**.

**Why this notebook exists.** The deck's *Confidence score algorithm* (slide 9) is
specified across slides 10–12 and demonstrated on slide 13. Implementing the
specification exactly as written and running it against the deck's own five
example stations does **not** reproduce the deck's answers: one row disagrees.

This notebook makes that reproducible so a partner or data expert can decide which
reading is correct. **It changes no product code** — the hub's current
implementation lives in `backend/api/v1/v1_weather/confidence.py` and is compared
in section 6.

---

### Slide map

| Slide | Title | What it gives us |
|---|---|---|
| 5 | The validation process | Step 2 compares **SPI deltas** (`SPI −2.1 sat vs −2.5 stn → Δ0.4 medium`) |
| 9 | Confidence score algorithm | The three steps and the 0.4 / 0.6 weighting |
| 10 | Temperature score | `abs(ΔT)` bands, satellite LST vs **station max** |
| 11 | Precipitation score | Dry-day branches + `abs(sat − stn) / stn × 100` bands |
| 12 | Combination of precipitation and temperature | Hard veto on 1, soft veto capping at 2, weighted average |
| 13 | Worked example | Five stations with expected scores — **the fixture below** |
| 14 | "Are these the right thresholds?" | The deck itself flags the cut-offs as temporary |

> Slide 14 states: *"The current proposed cut-offs are temporary, we need to agree
> on those before the framework goes operational."* This notebook is input to that
> agreement.


## 1 · The specification, transcribed verbatim

Nothing below is invented. Each constant and function cites the slide it comes
from, so a reviewer can check the transcription against the PDF line by line.


In [ ]:
import itertools
import math

import pandas as pd

# --- Slide 10 · Temperature score --------------------------------------------
# "Compute the absolute difference between the satellite LST reading and the
#  station max. Smaller differences -> higher confidence."
TEMPERATURE_BANDS = ((0.5, 5), (1.5, 4), (3.0, 3), (5.0, 2))      # else 1

# --- Slide 11 · Precipitation score ------------------------------------------
# Percentage-difference bands, applied after the dry-day branches.
PRECIPITATION_PCT_BANDS = ((10, 5), (25, 4), (50, 3), (100, 2))   # else 1

# --- Slide 12 · Combination ---------------------------------------------------
TEMPERATURE_WEIGHT, PRECIPITATION_WEIGHT = 0.4, 0.6
HARD_VETO, SOFT_VETO = 1, 2


def temperature_score(station_c, satellite_c):
    """Slide 10. |satellite LST - station max|, banded."""
    delta = abs(satellite_c - station_c)
    for ceiling, score in TEMPERATURE_BANDS:
        if delta <= ceiling:
            return score, delta
    return 1, delta


def precipitation_score(station_mm, satellite_mm, denominator="stn"):
    """Slide 11.

    The dry-day branches are unambiguous and transcribed as written. The
    `denominator` argument exists ONLY because slide 11's formula and slide 13's
    worked example disagree -- see section 3. "stn" is what slide 11 literally
    prints; the others are the candidate readings.
    """
    if station_mm == 0 and satellite_mm == 0:
        return 5, "both dry, full agreement"
    if station_mm == 0 and satellite_mm > 0:
        if satellite_mm <= 1.0:
            return 4, "tiny - likely noise"
        if satellite_mm <= 5.0:
            return 3, "localised shower missed by gauge"
        return 1, "large mismatch - review"

    base = {
        "stn": station_mm,                      # slide 11, exactly as printed
        "sat": satellite_mm,
        "max": max(station_mm, satellite_mm),   # symmetric relative difference
    }[denominator]
    pct = abs(satellite_mm - station_mm) / base * 100
    for ceiling, score in PRECIPITATION_PCT_BANDS:
        if pct <= ceiling:
            return score, f"{pct:.1f}%"
    return 1, f"{pct:.1f}%"


def combine(temperature, precipitation):
    """Slide 12. Hard veto on 1; soft veto caps at 2; else weighted average."""
    if temperature == HARD_VETO or precipitation == HARD_VETO:
        return 1, "hard veto on 1"
    weighted = (TEMPERATURE_WEIGHT * temperature
                + PRECIPITATION_WEIGHT * precipitation)
    # floor(x + 0.5), not round(): Python's round() is banker's rounding, so a
    # weighted 4.5 would fall to 4. Slide 13 writes 4.6 -> 5.
    combined = math.floor(weighted + 0.5)
    if temperature == SOFT_VETO or precipitation == SOFT_VETO:
        return min(SOFT_VETO, combined), "soft veto caps at 2"
    return combined, (f"{TEMPERATURE_WEIGHT}*{temperature} + "
                      f"{PRECIPITATION_WEIGHT}*{precipitation} = {weighted:.1f}")


## 2 · The worked example, transcribed from slide 13

Five stations with the scores the deck prints for each. `exp_*` is what the slide
asserts — the fixture we are trying to reproduce.


In [ ]:
WORKED_EXAMPLE = pd.DataFrame([
    # stn_C, sat_C, stn_mm, sat_mm, exp_t, exp_p, exp_combined, deck's own logic
    (25.0, 25.2, 10.0, 10.5, 5, 5, 5, "0.4*5 + 0.6*5 = 5 (no veto)"),
    (24.8, 26.0,  0.0,  0.0, 4, 5, 5, "0.4*4 + 0.6*5 = 4.6 -> 5"),
    (28.0, 29.5,  5.0,  8.0, 4, 3, 3, "0.4*4 + 0.6*3 = 3.4 -> 3"),
    (22.0, 27.5, 20.0,  2.0, 1, 2, 1, "temp score = 1 -> hard veto -> 1"),
    (26.5, 26.6,  0.0,  6.0, 5, 1, 1, "precip score = 1 -> hard veto -> 1"),
], columns=["station_c", "satellite_c", "station_mm", "satellite_mm",
            "exp_t", "exp_p", "exp_combined", "deck_logic"])

WORKED_EXAMPLE


## 3 · Running the specification as written

`denominator="stn"` is slide 11 exactly: `abs(sat − stn) / stn × 100`.


In [ ]:
def run(example, denominator):
    rows = []
    for i, r in example.iterrows():
        t, delta_t = temperature_score(r.station_c, r.satellite_c)
        p, basis = precipitation_score(r.station_mm, r.satellite_mm, denominator)
        c, logic = combine(t, p)
        rows.append({
            "#": i + 1,
            "|dT|": round(delta_t, 1),
            "T": t, "T ok": t == r.exp_t,
            "precip basis": basis,
            "P": p, "P ok": p == r.exp_p,
            "combined": c, "C ok": c == r.exp_combined,
            "our logic": logic,
            "deck says": f"({r.exp_t}, {r.exp_p}, {r.exp_combined})",
        })
    return pd.DataFrame(rows).set_index("#")


as_written = run(WORKED_EXAMPLE, "stn")
print("Reproduces slide 13 exactly:",
      bool(as_written[["T ok", "P ok", "C ok"]].all().all()))
as_written


### Row 3 does not reproduce

Station **5.0 mm** vs satellite **8.0 mm**:

- Slide 11's formula gives `abs(8 − 5) / 5 × 100` = **60%**
- Slide 11's own band table puts 60% in `<= 100% -> 2`
- Slide 13 assigns **3**

The disagreement then propagates: a precipitation score of 2 triggers slide 12's
**soft veto**, capping the combined score at 2, where the deck shows **3**.

Note the deck labels that cell "+60%" — so it computed the same percentage this
notebook does, then assigned a score its own band table does not allow.


## 4 · Which reading reproduces the deck?

Three candidate denominators for the relative difference. Only one is consistent
with all five rows.


In [ ]:
summary = {}
for denominator in ("stn", "sat", "max"):
    result = run(WORKED_EXAMPLE, denominator)
    matched = result[["T ok", "P ok", "C ok"]].all(axis=1)
    summary[denominator] = {
        "rows reproduced": f"{int(matched.sum())} / {len(result)}",
        "reproduces slide 13": bool(matched.all()),
    }

pd.DataFrame(summary).T


In [ ]:
# Row by row, so the disagreement is visible rather than asserted.
per_row = []
for denominator in ("stn", "sat", "max"):
    for i, r in WORKED_EXAMPLE.iterrows():
        p, basis = precipitation_score(r.station_mm, r.satellite_mm, denominator)
        per_row.append({"denominator": denominator, "#": i + 1,
                        "stn mm": r.station_mm, "sat mm": r.satellite_mm,
                        "deck P": r.exp_p, "P": p, "basis": basis})

pd.DataFrame(per_row).pivot_table(
    index=["#", "stn mm", "sat mm", "deck P"],
    columns="denominator", values="P",
).reset_index()


In [ ]:
# The reading that reproduces slide 13 in full.
run(WORKED_EXAMPLE, "max")


### Result

| Denominator | Row 1 | Row 3 | Row 4 | Reproduces slide 13 |
|---|---|---|---|---|
| `/ stn` — **as printed on slide 11** | ok | 60% → 2, deck says 3 | ok | **no** |
| `/ sat` | ok | 37.5% → 3 | 900% → 1, deck says 2 | **no** |
| `/ max(stn, sat)` | ok | 37.5% → 3 | 90% → 2 | **yes — all five rows** |

Under `max`, row 3 reproduces the deck's score *and* its printed arithmetic
(`0.4·4 + 0.6·3 = 3.4 → 3`). That is strong evidence the intended formula is the
**symmetric** relative difference, and that `/ stn` on slide 11 is a
transcription slip.

It is also the better-behaved formula: symmetric when satellite and station are
swapped, and it does not explode when the gauge reads close to zero.


## 5 · But the symmetric reading makes one band unreachable

This is why the question needs a data expert rather than a developer.


In [ ]:
# With both values positive, abs(sat - stn) / max(stn, sat) is bounded by 100%,
# so slide 11's ">100% -> 1" band can never be reached through the percentage
# path. A precipitation score of 1 would then come ONLY from the dry-gauge
# "sat > 5 mm" branch. Demonstrated over a sweep rather than argued:
values = [0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 250, 500]
reachable = {
    denominator: sorted({
        precipitation_score(stn, sat, denominator)[0]
        for stn, sat in itertools.product(values, values)
        if stn > 0                      # dry-gauge branch excluded on purpose
    })
    for denominator in ("stn", "sat", "max")
}

pd.DataFrame({
    "scores reachable via the % path": {k: str(v) for k, v in reachable.items()},
    "can the % path ever score 1": {k: (1 in v) for k, v in reachable.items()},
})


## 6 · The other open question: slide 5 vs slide 11

The deck describes precipitation agreement **twice, differently**:

- **Slide 5** (*The validation process*, Step 2) compares **SPI values**:
  `SPI = −2.1 (satellite) vs −2.5 (station)` → `Δ0.4 (Medium confidence)`
- **Slide 11** (*Precipitation score*) compares **millimetres** as a percentage
  difference, with dry-day branches

These are different algorithms over different quantities. The hub currently
implements the **slide 5** reading — see `backend/api/v1/v1_weather/confidence.py`,
whose banding constants cite that worked example by name.

**Why slide 11 was not implemented:** it needs satellite rainfall in millimetres,
and the CDI pipeline publishes only a percentile rank of `chirps_spi_3mn`. There
was no `sat` in mm to put in the formula.

**That constraint has since lifted.** `AdministrationObservation` now holds real
CHIRPS monthly totals in mm per Inkhundla (WX-10), so slide 11's algorithm is
implementable today, dry-day branches included.


In [ ]:
# Confirm the mm coverage for yourself against the live database:
#
#   docker compose exec -T backend ./manage.py shell -c \
#     "from api.v1.v1_weather.models import AdministrationObservation as A; \
#      print(A.objects.count(), list(A.objects.values('parameter','dataset').distinct()))"
#
# Measured 2026-09-02 on the local database:
#   649 rows | precipitation | CHIRPS v2.0 africa_monthly | 2025-09 .. 2026-07
#   = 59 Tinkhundla x 11 months, full coverage.

pd.DataFrame([
    ("Combination + vetoes", "hard 1 / soft 2 / 0.4t + 0.6p",
     "identical", "matches"),
    ("Temperature bands", "0.5 / 1.5 / 3.0 / 5.0 C",
     "TEMPERATURE_SCORE_BANDS, identical", "matches but never invoked"),
    ("Precipitation method", "% difference on monthly mm (slide 11)",
     "SPI z-delta over a 3-month window (slide 5)", "DIFFERENT ALGORITHM"),
    ("Dry-day branches", "four explicit cases", "none", "absent"),
    ("Score range", "1-5", "0-5, where 0 = not computable",
     "extension the deck does not cover"),
], columns=["element", "Validation Framework", "confidence.py", "status"])


### Why the temperature half never runs

Slide 10 assumes a satellite **LST** reading. The CDI pipeline replaced MODIS LST
with **ESI** (`STEP_0303_ESI_pct_rank_Eswatini_*`), a dimensionless evaporative
stress rank. There is no satellite temperature in degrees anywhere in the hub, so
`temperature` is always `None` and the combined score runs on precipitation alone.

The bands are implemented and correct; they simply have no input.


## 7 · Decisions needed

None of these can be settled in code. Each changes published confidence scores.

**Q1 — Slide 13 row 3 disagrees with slide 11. Which is authoritative?**

- **(a)** The formula is symmetric, `abs(sat − stn) / max(stn, sat)`, and slide
  11's `/ stn` is a slip. Then the `> 100% -> 1` band is unreachable and the
  cut-offs need re-cutting.
- **(b)** The formula is `/ stn` as printed, and row 3 of the worked example is
  wrong — it should read P = 2, combined = 2.
- **(c)** Something else was intended that neither reading captures.

**Q2 — Slide 5 or slide 11?** Should precipitation agreement be scored on SPI
deltas (what is built) or on a percentage difference of monthly millimetres (what
slide 11 specifies)? Both are now possible.

> This is not only a modelling preference. The slide 5 method needs a **3-month**
> window, and its 20-reporting-day floor is currently why **41 of 59 Tinkhundla**
> score 0 for 2026-06 and 2026-07 — May 2026 holds only 6 days of station data and
> cannot be backfilled, because the WIS2 archive begins 2026-05-26. Slide 11's
> method is **monthly**, so it would not need May at all.

**Q3 — Temperature.** Slide 10 compares satellite LST against the station **max**.
No satellite temperature exists. Options: leave the half switched off, or replace
it with what the hub does hold — satellite ESI rank alongside the station's
departure from its 30-year normal, shown side by side rather than differenced.

**Q4 — Thresholds.** Slide 14 already asks this. Any answer to Q1 or Q2 changes
which cut-offs are being agreed to.
